In [8]:
import pandas as pd
import numpy as np
import math
import re
from pathlib import Path
from urllib.parse import urlparse

# Set display options to see all columns
pd.set_option('display.max_columns', None)
print("Environment Ready.")

Environment Ready.


In [9]:
df = pd.read_csv(Path.cwd().parent /'master_dataset_clean.csv')

df.head()

,url,label
0,noxlogic.nl,0
1,dubaiexch.live,0
2,brightroulettegroup.com,0
3,worldglobalmedia.co.uk,0
4,fobo-friends.ru,0


In [10]:
def get_entropy(text):
    """Calculates the Shannon Entropy (randomness) of a string."""
    if not text: 
        return 0
    # Calculate frequency of each character
    probabilities = [n_x/len(text) for n_x in pd.Series(list(text)).value_counts()]
    # Calculate Shannon Entropy
    entropy = -sum(p * math.log2(p) for p in probabilities)
    return entropy

def is_ip(hostname: str) -> int:
    """Checks if the hostname is a raw IPv4 address."""
    host = hostname.split(":")[0]  # strip port if present
    ipv4_pattern = re.compile(r"^\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}$")
    return 1 if ipv4_pattern.match(host) else 0

print("Helper functions defined.")

Helper functions defined.


In [11]:
SUSPICIOUS_TLDS = {
    "tk","ml","ga","cf","gq","xyz","top","click","link","work",
    "date","faith","review","party","science","cricket","bid",
    "loan","win","racing","download","accountant","trade","webcam",
    "cfd","wiki","zip","mov"
}

BRAND_KEYWORDS = [
    "paypal","google","apple","amazon","microsoft","facebook","instagram",
    "netflix","bank","secure","account","update","verify","login","signin"
]

URL_SHORTENERS = {
    "bit.ly","tinyurl.com","goo.gl","t.co","ow.ly","buff.ly",
    "shorturl.at","is.gd","rb.gy","cutt.ly"
}

def extract_lexical_features(url):
    """Breaks down a URL into numerical features."""
    features = {}
    url_str = str(url)

    # Prepend scheme only for urlparse component splitting, never stored
    temp = ("http://" + url_str) if not url_str.startswith("http") else url_str
    parsed = urlparse(temp)
    hostname = parsed.netloc.split(":")[0]   # strip port
    path     = parsed.path
    parts    = hostname.split(".")
    tld      = parts[-1].lower() if parts else ""

    # ── Original 9 features (preserved) ──────────────────────────────────
    features['url_length']      = len(url_str)
    features['hostname_length'] = len(hostname)
    features['dot_count']       = url_str.count('.')
    features['hyphen_count']    = url_str.count('-')
    features['at_count']        = url_str.count('@')
    features['slash_count']     = url_str.count('/')
    features['query_count']     = url_str.count('?')
    features['is_ip']           = is_ip(hostname)
    features['url_entropy']     = get_entropy(url_str)

    # ── 7 New features ────────────────────────────────────────────────────
    features['subdomain_count']    = max(len(parts) - 2, 0)
    features['suspicious_tld']     = int(tld in SUSPICIOUS_TLDS)
    features['digit_ratio']        = sum(c.isdigit() for c in hostname) / max(len(hostname), 1)
    features['has_port']           = int(":" in parsed.netloc)
    features['path_depth']         = path.count('/')
    features['brand_in_subdomain'] = int(any(b in hostname.lower() for b in BRAND_KEYWORDS))
    features['is_url_shortener']   = int(hostname.lower() in URL_SHORTENERS)

    return features

print("Extraction engine ready.")

Extraction engine ready.


In [ ]:
print("Extracting features... Please wait.")

df['url'] = df['url'].astype(str)

def safe_extract(url):
    try:
        return extract_lexical_features(url)
    except Exception:
        return {
            'url_length': 0, 'hostname_length': 0,
            'dot_count': 0, 'hyphen_count': 0, 'at_count': 0,
            'slash_count': 0, 'query_count': 0,
            'is_ip': 0, 'url_entropy': 0,
            'subdomain_count': 0, 'suspicious_tld': 0,
            'digit_ratio': 0, 'has_port': 0,
            'path_depth': 0, 'brand_in_subdomain': 0,
            'is_url_shortener': 0
        }

feature_list = df['url'].apply(safe_extract).tolist()

feature_df = pd.DataFrame(feature_list)
df_final = pd.concat([df, feature_df], axis=1)

df_final = df_final[df_final['url_length'] > 0]

print(f"Extraction complete. Rows after cleaning: {len(df_final)}")
df_final.head()
(df_final[df_final['label'] == 1].head(5))

Extracting features... Please wait.
Extraction complete. Rows after cleaning: 585034


,url,label,url_length,hostname_length,dot_count,hyphen_count,at_count,slash_count,query_count,is_ip,url_entropy,subdomain_count,suspicious_tld,digit_ratio,has_port,path_depth,brand_in_subdomain,is_url_shortener
0,noxlogic.nl,0,11,11,1,0,0,0,0,0,2.913977,0,0,0.0,0,0,0,0
1,dubaiexch.live,0,14,14,1,0,0,0,0,0,3.521641,0,0,0.0,0,0,0,0
2,brightroulettegroup.com,0,23,23,1,0,0,0,0,0,3.642490,0,0,0.0,0,0,0,0
3,worldglobalmedia.co.uk,0,22,22,2,0,0,0,0,0,3.754442,1,0,0.0,0,0,0,0
4,fobo-friends.ru,0,15,15,1,1,0,0,0,0,3.506891,0,0,0.0,0,0,0,0


In [17]:
(df_final[df_final['label'] == 1].head(5))

,url,label,url_length,hostname_length,dot_count,hyphen_count,at_count,slash_count,query_count,is_ip,url_entropy,subdomain_count,suspicious_tld,digit_ratio,has_port,path_depth,brand_in_subdomain,is_url_shortener
300000,112.93.202.244:48456/i,1,22,14,3,0,0,1,0,1,3.425119,2,0,0.785714,1,1,0,0
300001,221.1.227.205:35291/i,1,21,13,3,0,0,1,0,1,3.105672,2,0,0.769231,1,1,0,0
300002,binary-block-tabel-expert-get.wiki/92c764c1-ff...,1,81,34,2,8,0,2,0,0,4.654622,0,1,0.000000,0,2,0,0
300003,112.93.202.244:48456/bin.sh,1,27,14,4,0,0,1,0,1,3.791925,2,0,0.785714,1,1,0,0
300004,binary-block-state-collection.wiki/f9676fa8-b8...,1,81,34,2,7,0,2,0,0,4.637819,0,1,0.000000,0,2,0,0


In [13]:
# Save to CSV for the Model Training phase
df_final.to_csv(Path.cwd().parent /'master_dataset_lexical_features.csv', index=False)

print("Success! File saved as 'master_dataset_lexical_features.csv'.")
print(f"Final Column List: {df_final.columns.tolist()}")

Success! File saved as 'master_dataset_lexical_features.csv'.
Final Column List: ['url', 'label', 'url_length', 'hostname_length', 'dot_count', 'hyphen_count', 'at_count', 'slash_count', 'query_count', 'is_ip', 'url_entropy', 'subdomain_count', 'suspicious_tld', 'digit_ratio', 'has_port', 'path_depth', 'brand_in_subdomain', 'is_url_shortener']


In [14]:
# CHECKING

# After extraction, check what safe URLs look like
safe_sample = df_final[df_final['label'] == 0].head(5)
print("Safe URL features:")
print(safe_sample[['url', 'slash_count',  'hostname_length']].to_string())

phish_sample = df_final[df_final['label'] == 1].head(5)
print("\nPhishing URL features:")
print(phish_sample[['url', 'slash_count', 'hostname_length']].to_string())

Safe URL features:
                       url  slash_count  hostname_length
0              noxlogic.nl            0               11
1           dubaiexch.live            0               14
2  brightroulettegroup.com            0               23
3   worldglobalmedia.co.uk            0               22
4          fobo-friends.ru            0               15

Phishing URL features:
                                                                                      url  slash_count  hostname_length
300000                                                             112.93.202.244:48456/i            1               14
300001                                                              221.1.227.205:35291/i            1               13
300002  binary-block-tabel-expert-get.wiki/92c764c1-ff82-4023-a702-309971bc7633/google.ct            2               34
300003                                                        112.93.202.244:48456/bin.sh            1               14
300004  binary-